In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch.utils.data as data
from torch.utils.data import Subset
import torch_pruning as tp
from ptflops import get_model_complexity_info

ModuleNotFoundError: No module named 'torch_pruning'

# Preparations

## Simple CNN for MNIST

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.fc1 = nn.Linear(12*12*64, 128)  # 28x28 input -> 24x24 after conv1 -> 22x22 after conv2 -> 11x11 after pooling(2)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = nn.functional.relu(self.conv1(x))
        x = nn.functional.relu(self.conv2(x))
        x = nn.functional.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = nn.functional.relu(self.fc1(x))
        x = self.fc2(x)
        return x

## Helper Functions

In [ ]:
# Calculate FLOPs and parameters (adjusted for 1-channel input)
def print_model_flops(model, input_res=(1, 28, 28), name="Model"):
    macs, params = get_model_complexity_info(model, input_res, as_strings=True,
                                             print_per_layer_stat=False, verbose=False)
    print(f"{name} – FLOPs: {macs}, Parameters: {params}")

# Calculate test accuracy
def evaluate_model(model, dataloader, device, name="Model"):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    print(f"{name} – Test Accuracy: {accuracy:.2f}%")
    return accuracy


## Training functions

In [ ]:
# Training for a given number of epochs
def train_epochs(model, device, dataloader, epochs=1):
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(epochs):
        for batch_idx, (inputs, labels) in enumerate(dataloader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            if batch_idx % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs} - Loss: {loss.item():.4f}")
    print(f"Training for {epochs} epoch(s) completed.\n")

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# MNIST transforms (no resizing needed, only normalization)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])


## Dataset

In [ ]:
# MNIST train and test datasets (small subsets)
trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_subset = Subset(trainset, range(500))  # 500 images for training
trainloader = data.DataLoader(train_subset, batch_size=64, shuffle=True)

testset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_subset = Subset(testset, range(100))  # 100 images for testing
testloader = data.DataLoader(test_subset, batch_size=64, shuffle=False)


## Training and Evaluation

In [ ]:
# Prepare model
model = SimpleCNN().to(device)

# FLOPs & Accuracy before pruning
print_model_flops(model, name="Before Pruning")
evaluate_model(model, testloader, device, name="Before Pruning")

# Example input for pruning
example_inputs = torch.randn(1, 1, 28, 28).to(device)

# Pruning setup
importance = tp.importance.MagnitudeImportance(p=1)  # L1 norm

# AGP setup: start at 10%, end at 50% sparsity over 3 steps
num_pruning_steps = 3
epochs_per_step = 5
initial_ratio = 0.1
final_ratio = 0.5
ratios = torch.linspace(initial_ratio, final_ratio, steps=num_pruning_steps)

for step, ratio in enumerate(ratios):
    print(f"=== Pruning Step {step+1}/{num_pruning_steps} (Ratio: {ratio:.2f}) ===")

    pruner = tp.pruner.MagnitudePruner(
        model=model,
        example_inputs=example_inputs,
        importance=importance,
        pruning_ratio=float(ratio),
        ignored_layers=[model.fc2],  # Do not prune the last layer
    )

    train_epochs(model, device, trainloader, epochs=epochs_per_step)
    pruner.step()
    print_model_flops(model, name=f"After Pruning Step {step+1}")
    evaluate_model(model, testloader, device, name=f"After Pruning Step {step+1}")

print("Final fine-tuning after all pruning steps...")
train_epochs(model, device, trainloader, epochs=5)
evaluate_model(model, testloader, device, name="Final Model")

print("Automated Gradual Pruning (AGP) + Fine-Tuning completed!")
